In [ ]:
!pip install torch torchvision torchaudio
!pip install sentencepiece
!pip install nltk bert-score

import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import sentencepiece as spm
import time
from nltk.translate.bleu_score import corpus_bleu
from bert_score import score as bert_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 1. Data Preparation ---
train_sa = pd.read_csv("/home/train_sa_10000.csv")
train_en = pd.read_csv("/home/train_en_10000.csv")
dev_sa   = pd.read_csv("/home/dev_sa_1000.csv")
dev_en   = pd.read_csv("/home/dev_en_1000.csv")
test_sa  = pd.read_csv("/home/test_sa_1000.csv")

train_pairs = pd.merge(train_sa, train_en, on="Source_id")
dev_pairs   = pd.merge(dev_sa, dev_en, on="Source_id")

# Helper to save clean text files for SentencePiece training
def save_clean_txt(df, col_name, filepath):
    with open(filepath, "w", encoding="utf-8") as f:
        for text in df[col_name].dropna().astype(str):
            f.write(text + "\n")

save_clean_txt(train_pairs, "Sentence_sa", "train_sa_clean.txt")
save_clean_txt(train_pairs, "Sentence_en", "train_en_clean.txt")

# Train tokenizers using clean text files
spm.SentencePieceTrainer.Train(
    "--input=train_sa_clean.txt --model_prefix=sp_sa --vocab_size=8000 "
    "--pad_id=0 --bos_id=1 --bos_piece=<s> --eos_id=2 --eos_piece=</s> --unk_id=3 --unk_piece=<unk>"
)
spm.SentencePieceTrainer.Train(
    "--input=train_en_clean.txt --model_prefix=sp_en --vocab_size=8000 "
    "--pad_id=0 --pad_piece=<pad> --bos_id=1 --bos_piece=<s> --eos_id=2 --eos_piece=</s> --unk_id=3 --unk_piece=<unk>"
)

sp_sa = spm.SentencePieceProcessor(model_file="sp_sa.model")
sp_en = spm.SentencePieceProcessor(model_file="sp_en.model")

# --- 2. Dataset and Loss Setup ---
class LabelSmoothingLoss(nn.Module):
    def __init__(self, classes, smoothing=0.1, ignore_index=0):
        super(LabelSmoothingLoss, self).__init__()
        self.classes = classes
        self.smoothing = smoothing
        self.ignore_index = ignore_index

    def forward(self, pred, target):
        confidence = 1.0 - self.smoothing
        true_dist = torch.zeros_like(pred)
        true_dist.fill_(self.smoothing / (self.classes - 1))
        mask = target != self.ignore_index
        true_dist.scatter_(1, target.unsqueeze(1), confidence)
        true_dist = true_dist * mask.unsqueeze(1)
        loss = torch.mean(torch.sum(-true_dist * F.log_softmax(pred, dim=1), dim=1))
        return loss

class TranslationDataset(Dataset):
    def __init__(self, df, sp_src, sp_tgt):
        self.df = df
        self.sp_src = sp_src
        self.sp_tgt = sp_tgt

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        src = self.sp_src.encode(self.df.iloc[idx]["Sentence_sa"], out_type=int)
        tgt = self.sp_tgt.encode(self.df.iloc[idx]["Sentence_en"], out_type=int)
        return torch.tensor(src), torch.tensor(tgt)

def collate_fn(batch):
    srcs, tgts = zip(*batch)
    srcs = nn.utils.rnn.pad_sequence(srcs, batch_first=True, padding_value=0)
    tgts = nn.utils.rnn.pad_sequence(tgts, batch_first=True, padding_value=0)
    return srcs, tgts

# FIXED: shuffle=True for the training DataLoader
train_loader = DataLoader(TranslationDataset(train_pairs, sp_sa, sp_en), batch_size=32, shuffle=True, collate_fn=collate_fn)
dev_loader   = DataLoader(TranslationDataset(dev_pairs, sp_sa, sp_en), batch_size=32, shuffle=False, collate_fn=collate_fn)

# --- 3. Model Architecture ---
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.rnn = nn.GRU(emb_dim, hid_dim, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(hid_dim*2, hid_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        emb = self.dropout(self.embedding(src))
        outputs, hidden = self.rnn(emb)
        hidden = torch.tanh(self.fc(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)))
        return outputs, hidden.unsqueeze(0)

class Attention(nn.Module):
    def __init__(self, hid_dim):
        super().__init__()
        self.attn = nn.Linear(hid_dim*3, hid_dim)
        self.v = nn.Linear(hid_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        src_len = encoder_outputs.size(1)
        hidden = hidden[-1].unsqueeze(1).repeat(1, src_len, 1)
        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        attention = self.v(energy).squeeze(2)
        return F.softmax(attention, dim=1)

class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, attention, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.rnn = nn.GRU(hid_dim*2 + emb_dim, hid_dim, batch_first=True)
        self.fc_out = nn.Linear(hid_dim*3 + emb_dim, vocab_size)
        self.attention = attention
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, encoder_outputs):
        input = input.unsqueeze(1)
        emb = self.dropout(self.embedding(input))
        attn_weights = self.attention(hidden, encoder_outputs).unsqueeze(1)
        context = torch.bmm(attn_weights, encoder_outputs)
        rnn_input = torch.cat((emb, context), dim=2)
        output, hidden = self.rnn(rnn_input, hidden)

        pred = self.fc_out(torch.cat((output.squeeze(1), context.squeeze(1), emb.squeeze(1)), dim=1))
        return pred, hidden

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        batch_size = src.shape[0]
        tgt_len = tgt.shape[1]
        tgt_vocab_size = self.decoder.embedding.num_embeddings
        outputs = torch.zeros(batch_size, tgt_len, tgt_vocab_size).to(device)
        encoder_outputs, hidden = self.encoder(src)
        input = tgt[:, 0]
        for t in range(1, tgt_len):
            output, hidden = self.decoder(input, hidden, encoder_outputs)
            outputs[:, t, :] = output
            teacher_force = torch.rand(1).item() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input = tgt[:, t] if teacher_force else top1
        return outputs

INPUT_DIM = len(sp_sa)
OUTPUT_DIM = len(sp_en)
HID_DIM = 512
EMB_DIM = 300

attn = Attention(HID_DIM)
enc = Encoder(INPUT_DIM, EMB_DIM, HID_DIM)
dec = Decoder(OUTPUT_DIM, EMB_DIM, HID_DIM, attn)
model = Seq2Seq(enc, dec).to(device)

criterion = LabelSmoothingLoss(classes=len(sp_en), smoothing=0.1, ignore_index=0)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=3, factor=0.5)

# --- 4. Training Function ---
def train(model, loader, epoch, total_epochs):
    model.train()
    epoch_loss = 0
    for src, tgt in loader:
        teacher_forcing_ratio = max(0.2, 0.7 * (1 - epoch/total_epochs))
        src, tgt = src.to(device), tgt.to(device)
        optimizer.zero_grad()
        output = model(src, tgt, teacher_forcing_ratio=teacher_forcing_ratio)
        output_dim = output.shape[-1]
        output = output[:, 1:].reshape(-1, output_dim)
        tgt = tgt[:, 1:].reshape(-1)
        loss = criterion(output, tgt)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1)
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(loader)

for epoch in range(10):
    loss = train(model, train_loader, epoch, 10)
    scheduler.step(loss)
    print(f"Epoch {epoch+1}, Loss={loss:.4f}")

# --- 5. FIXED Beam Search with EOS Tracking ---
def beam_search_decode(model, src, beam_width=5, max_len=50):
    model.eval()
    with torch.no_grad():
        encoder_outputs, hidden = model.encoder(src)
        # Form structure: [sequence_tokens, score, current_hidden]
        sequences = [[[], 0.0, hidden]]
        completed_sequences = []

        for _ in range(max_len):
            all_candidates = []
            for seq, score, current_hidden in sequences:
                # Early stop boundary check for active beam elements
                if seq and seq[-1] == 2:
                    completed_sequences.append([seq, score])
                    continue

                input_tok = torch.tensor([seq[-1]] if seq else [1]).to(device)  # Default <s>
                output, next_hidden = model.decoder(input_tok, current_hidden, encoder_outputs)
                probs = F.log_softmax(output, dim=1)
                topk = torch.topk(probs, beam_width)

                for i in range(beam_width):
                    candidate = [seq + [topk.indices[0][i].item()],
                                 score + topk.values[0][i].item(),
                                 next_hidden]
                    all_candidates.append(candidate)

            if not all_candidates:
                break

            ordered = sorted(all_candidates, key=lambda x: x[1], reverse=True)
            sequences = ordered[:beam_width - len(completed_sequences)]

            if len(sequences) == 0:
                break

        all_final = completed_sequences + [[s[0], s[1]] for s in sequences]
        all_final = sorted(all_final, key=lambda x: x[1], reverse=True)

        # Extract and strip out the control characters (<s>, </s>, <pad>)
        final_tokens = all_final[0][0]
        return [t for t in final_tokens if t not in [0, 1, 2]]

def translate_sentence(sentence):
    tokens = sp_sa.encode(sentence, out_type=int)
    src = torch.tensor(tokens).unsqueeze(0).to(device)
    pred_tokens = beam_search_decode(model, src)
    return sp_en.decode(pred_tokens)

# --- 6. Evaluation Functions with Safe Cleanups ---
def evaluate_bleu(model, loader):
    model.eval()
    refs, hyps = [], []
    with torch.no_grad():
        for src, tgt in loader:
            src, tgt = src.to(device), tgt.to(device)
            for i in range(src.size(0)):
                pred_tokens = beam_search_decode(model, src[i].unsqueeze(0))
                pred_sentence = sp_en.decode(pred_tokens)

                # FIXED: Stripping index control tokens before running BLEU alignment evaluation
                clean_tgt = [tok for tok in tgt[i].cpu().numpy().tolist() if tok not in [0, 1, 2]]
                ref_sentence = sp_en.decode(clean_tgt)

                hyps.append(pred_sentence.split())
                refs.append([ref_sentence.split()])
    return corpus_bleu(refs, hyps)

def evaluate_bertscore(model, loader):
    model.eval()
    refs, hyps = [], []
    with torch.no_grad():
        for src, tgt in loader:
            src, tgt = src.to(device), tgt.to(device)
            for i in range(src.size(0)):
                pred_tokens = beam_search_decode(model, src[i].unsqueeze(0))
                pred_sentence = sp_en.decode(pred_tokens)

                # FIXED: Strip padding array references out
                clean_tgt = [tok for tok in tgt[i].cpu().numpy().tolist() if tok not in [0, 1, 2]]
                ref_sentence = sp_en.decode(clean_tgt)

                hyps.append(pred_sentence)
                refs.append(ref_sentence)
    P, R, F1 = bert_score(hyps, refs, lang="en", rescale_with_baseline=True)
    return F1.mean().item()

def measure_efficiency(model, test_df):
    start = time.time()
    preds = []
    with torch.no_grad():
        for i in range(len(test_df)):
            src = sp_sa.encode(test_df.iloc[i]["Sentence_sa"], out_type=int)
            src = torch.tensor(src).unsqueeze(0).to(device)
            pred_tokens = beam_search_decode(model, src)
            pred_sentence = sp_en.decode(pred_tokens)
            preds.append((test_df.iloc[i]["Source_id"], pred_sentence, test_df.iloc[i]["Sentence_sa"]))
    end = time.time()
    return preds, end - start, sum(p.numel() for p in model.parameters())

# --- 7. Execution and Output Tracking ---
bleu_score = evaluate_bleu(model, dev_loader)
print("BLEU (Dev):", bleu_score)

bert_f1 = evaluate_bertscore(model, dev_loader)
print("BERTScore F1 (Dev):", bert_f1)

preds, inf_time, num_params = measure_efficiency(model, test_sa)
print("Inference Time (s):", inf_time)
print("Total Parameters:", num_params)

submission = pd.DataFrame(preds, columns=["Source_id", "Sentence_en", "Sentence_sa"])
submission.to_csv("submission.csv", index=False, encoding="utf-8")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.7 MB/s eta 0:00:00
Epoch 1, Loss=2.5773
Epoch 2, Loss=2.3948
Epoch 3, Loss=2.2925
Epoch 4, Loss=2.2892
